In [10]:
# Task 1: Fetch Data from API
# This section will handle fetching data from the HackerNews API, categorizing stories based on keywords, and preparing the data for storage.

# Cell 1: Imports and Global Constants
import requests
import time
import json
import os
from datetime import datetime

# API Endpoints
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_DETAILS_URL_TEMPLATE = "https://hacker-news.firebaseio.com/v0/item/{id}.json"

# Request Headers
HHEADERS = {"User-Agent": "TrendPulse/1.0"}

# Category Definitions
CATEGORIES = {
    "technology": ["AI", "software", "tech", "code", "computer", "data", "cloud", "API", "GPU", "LLM"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["NFL", "NBA", "FIFA", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "NASA", "genome"],
    "entertainment": ["movie", "film", "music", "Netflix", "game", "book", "show", "award", "streaming"]
}

# Limits
MAX_TOP_STORY_IDS_TO_FETCH = 500
MAX_STORIES_PER_CATEGORY = 100 # Increased significantly to ensure at least 100 total stories are collected if available

# Cell 2: Helper Function to Fetch Top Story IDs
def fetch_top_story_ids():
    """Fetches the top story IDs from HackerNews API."""
    print(f"Fetching top {MAX_TOP_STORY_IDS_TO_FETCH} story IDs...")
    try:
        response = requests.get(TOP_STORIES_URL, headers=HEADERS)
        response.raise_for_status()  # Raise an exception for HTTP errors
        top_ids = response.json()
        return top_ids[:MAX_TOP_STORY_IDS_TO_FETCH]
    except requests.exceptions.RequestException as e:
        print(f"Error fetching top story IDs: {e}")
        return []

# Cell 3: Helper Function to Fetch Story Details
def fetch_story_details(story_id):
    """Fetches details for a given story ID from HackerNews API."""
    url = ITEM_DETAILS_URL_TEMPLATE.format(id=story_id)
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching details for story ID {story_id}: {e}")
        return None

# Cell 4: Helper Function to Categorize a Story
def get_story_category(title):
    """Determines the category of a story based on its title and predefined keywords."""
    if not title:
        return "uncategorized"

    title_lower = title.lower()
    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            if keyword.lower() in title_lower:
                return category
    return "uncategorized"

# Main Data Collection Logic
# This cell orchestrates the fetching, categorization, and extraction of story data.

# Cell 5: Main Logic to Collect and Process Stories

all_collected_stories = []
category_counts = {category: 0 for category in CATEGORIES}

top_story_ids = fetch_top_story_ids()

print("Starting story collection...")
stories_processed = 0

for story_id in top_story_ids:
    # Stop if at least 100 stories have been collected to meet the minimum requirement
    if len(all_collected_stories) >= 100:
        print("Minimum 100 stories collected. Stopping collection.")
        break

    # Check if all categories are full based on the new MAX_STORIES_PER_CATEGORY
    if all(count >= MAX_STORIES_PER_CATEGORY for count in category_counts.values()):
        print("All categories filled. Stopping collection.")
        break

    story_details = fetch_story_details(story_id)
    if not story_details or story_details.get('deleted') or story_details.get('dead'):
        continue

    title = story_details.get('title')
    if not title:
        continue

    category = get_story_category(title)

    if category != "uncategorized" and category_counts[category] < MAX_STORIES_PER_CATEGORY:
        collected_at = datetime.now().isoformat()
        story_data = {
            "post_id": story_details.get('id'),
            "title": title,
            "category": category,
            "score": story_details.get('score', 0),
            "num_comments": story_details.get('descendants', 0),
            "author": story_details.get('by', 'unknown'),
            "collected_at": collected_at
        }
        all_collected_stories.append(story_data)
        category_counts[category] += 1
        stories_processed += 1

        print(f"Collected story '{title}' ({category}). Total collected: {len(all_collected_stories)}")

        # Implement the 2-second delay only when a category is filled
        if category_counts[category] == MAX_STORIES_PER_CATEGORY:
            print(f"Category '{category}' has reached its limit ({MAX_STORIES_PER_CATEGORY} stories). Waiting 2 seconds before continuing...")
            time.sleep(2)

print(f"Finished story collection. Total stories collected: {len(all_collected_stories)}")
print(f"Stories collected per category: {category_counts}")

# Save Data to JSON File
# This cell handles saving the collected stories to a JSON file in the `data/` directory.

# Cell 6: Save to JSON File

DATA_DIR = "data"

# Create data directory if it doesn't exist
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Generate filename with current date
current_date = datetime.now().strftime("%Y%m%d")
filename = f"{DATA_DIR}/trends_{current_date}.json"

# Save all stories to JSON file
try:
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_collected_stories, f, indent=4, ensure_ascii=False)
    print(f"Collected {len(all_collected_stories)} stories. Saved to {filename}")
except IOError as e:
    print(f"Error saving data to {filename}: {e}")

Fetching top 500 story IDs...
Starting story collection...
Collected story 'Cloud in a Bottle: making self-hosting accessible to everyone' (technology). Total collected: 1
Collected story 'Chrome again exempts Google from user site data settings' (technology). Total collected: 2
Collected story 'Discovery of a new OpenAI agent message board' (technology). Total collected: 3
Collected story 'Visualizing Rust's Vtables: How dyn Trait Works In Memory' (technology). Total collected: 4
Collected story 'LLMs as a Cognitive Virus' (technology). Total collected: 5
Collected story 'Music Theory for Programmers' (entertainment). Total collected: 6
Collected story 'AI, Tools and Transformation' (technology). Total collected: 7
Collected story 'Show HN: Fly By – retro biplane flying game' (sports). Total collected: 8
Collected story 'OKF Agent Memory – Git-native persistent memory for AI coding agents' (technology). Total collected: 9
Collected story 'Can AI design circuit boards yet?' (technology